In [ ]:
##langchain components to use
from langchain.vectorstore.cassandra import Cassandra
from langchain.indexes.vectorstore import VectorStoreIndexWrapper
from langchain.embeddings import OllamaEmbeddings
from langchain.llms import Ollama 



#from langchain.llm import OpenAI
#from langchain.embeddings import OpenAIEmbeddings
#support for datasets retreival with hugging face
from datasets import load_dataset

##with cassio, the engine powering  the ASTRA DB integration in langchain 
##you will also initialize the db connection
import cassio


In [ ]:
!pip install pypdf2

In [ ]:
from pypdf2 import PdfReader

setup

In [ ]:
Astra_db_application_token="token"  
astra_db_id="astra_db_id"
##openai api key=""

In [ ]:
##provide the path of pdf file/files
PdfReader=PdfReader("budget_pdf")


In [ ]:
##divide content into some kind of chunks
from typing_extensions import Concatenate
#read text from pdf
raw_text=""
for i,page in enumerate(PdfReader.pages):
    content=page.extract_text()
    if content:
        raw_text+=content




In [ ]:
raw_text  ## it will have entie text from pdf


In [ ]:
cassio.init(token=Astra_db_application_token,db_id=astra_db_id)

##create the langchain embedding and llm objects for later usage

In [ ]:
llm=Ollama(model="llama2",temperature=0.7)
embedding=OllamaEmbeddings(model="llama2")

In [ ]:
astra_vector_store = Cassandra(
    embedding=embedding,
    table_name="qa_mini_demo",
    session=None,
    keyspace=None,
)



In [ ]:
from langchain.text_splitter import CharacterTextSplitter
# We need to split the text using Character Text Split such that it should not increase too much
text_splitter = CharacterTextSplitter(
    separator = "\n",
    chunk_size = 800,
    chunk_overlap = 200,
    length_function = len,
)
texts = text_splitter.split_text(raw_text)

In [ ]:
##load the dataset into the vector store
astra_vector_store.add_texts(texts[:50])
print("inserted headlines ," %len(texts[:50]))
astra_vector_index=VectorStoreIndexWrapper(vectorstore=astra_vector_store)

In [ ]:
first_question = True
while True:
    if first_question:
        query_text = input("\nEnter your question (or type 'quit' to exit): ").strip()
    else:
        query_text = input("\nWhat's your next question (or type 'quit' to exit): ").strip()

    if query_text.lower() == "quit":
        break

    if query_text == "":
        continue

    first_question = False

    print("\nQUESTION: \"%s\"" % query_text)
    answer = astra_vector_index.query(query_text, llm=llm).strip()